In [0]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import PipelineModel
import mlflow

df_for_ml = spark.table("workspace.gold.power_risk_features") 

pipeline_model = PipelineModel.load("/Volumes/workspace/default/my_volume/weather_pipeline")

train_ready = pipeline_model.transform(df_for_ml)
train_df, test_df = train_ready.randomSplit([0.8, 0.2], seed=42)

rf = RandomForestClassifier(labelCol="is_high_risk", featuresCol="features", numTrees=100)
rf_model = rf.fit(train_df)

In [0]:
predictions = rf_model.transform(test_df)
display(predictions.select("is_high_risk", "prediction", "probability"))

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="is_high_risk", rawPredictionCol="probability", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)

print(f"Model AUC Score: {auc}")

In [0]:
import pandas as pd

importances = rf_model.featureImportances.toArray()
feature_list = ["temperature", "windspeed", "cloudcover", "hour", "month", "day_of_week"]

importance_df = pd.DataFrame({'Feature': feature_list, 'Importance': importances})
print(importance_df.sort_values(by='Importance', ascending=False))

In [0]:
from pyspark.ml.linalg import Vectors

scenario_data = [(-10.0, 5.0, 100.0, 18.0, 1.0, 1.0)]
scenario_df = spark.createDataFrame(scenario_data, ["temperature_2m", "windspeed_10m", "cloudcover", "hour", "month", "day_of_week"])

scenario_vector = pipeline_model.transform(scenario_df)

final_prediction = rf_model.transform(scenario_vector)

display(final_prediction.select("probability", "prediction"))

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

importances = rf_model.featureImportances.toArray()
feature_list = ["temperature_2m", "windspeed_10m", "cloudcover", "hour", "month", "day_of_week"]

feat_imp_df = pd.DataFrame({'Feature': feature_list, 'Importance': importances})
feat_imp_df = feat_imp_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color='skyblue')
plt.xlabel('Importance Score')
plt.title('What drives the Blackout Risk?')
plt.gca().invert_yaxis()
plt.show()

In [0]:
import mlflow
import mlflow.spark
from mlflow.models.signature import infer_signature
from pyspark.sql import functions as F

volume_temp_path = "/Volumes/workspace/default/my_volume/mlflow_tmp"

feature_cols = ["temperature_2m", "windspeed_10m", "cloudcover", "hour", "month", "day_of_week"]
input_example = train_df.select([F.col(c).cast("double") for c in feature_cols]).limit(5).toPandas()

output_example = rf_model.transform(train_df).select("prediction").limit(5).toPandas()

signature = infer_signature(input_example, output_example)

with mlflow.start_run(run_name="Blackout_Risk_Model") as run:
    mlflow.log_param("num_trees", 100)
    mlflow.log_metric("auc", auc)
    
    mlflow.spark.log_model(
        spark_model=rf_model, 
        artifact_path="random-forest-model",
        dfs_tmpdir=volume_temp_path,
        signature=signature 
    )
    final_run_id = run.info.run_id
    
    print(f"Model successfully saved! Run ID: {final_run_id}")

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
uc_model_name = "workspace.default.power_risk_model_production"

model_details = mlflow.register_model(
    model_uri=f"runs:/{final_run_id}/random-forest-model",
    name=uc_model_name
)

client.set_registered_model_alias(uc_model_name, "champion", model_details.version)

print(f"✅ Success! Model version {model_details.version} is now tagged as 'champion'.")